<a href="https://colab.research.google.com/github/vmyel/thesis_pd/blob/main/pd_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 0.  Mount Google Drive & install/import libraries
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1.  USER-DEFINED PATHS  –– adjust to your Drive layout
# ============================================================
METADATA_PATH = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_files/corpus_PaHaW.xlsx'
SVC_ROOT      = '/content/drive/MyDrive/teasis/PaHaW/PaHaW_public/'   # folder that contains 00008/, 00009/, … sub-folders

# ============================================================
# 2.  Load & inspect metadata
# ============================================================
meta = pd.read_excel(METADATA_PATH, dtype={'ID': str})

# Normalise column names (strip whitespace, consistent case)
meta.columns = meta.columns.str.strip()

# Make sure the subject ID column is zero-padded to 5 chars
# (adjust 'ID' to whatever the actual column header is)
meta['ID'] = meta['ID'].astype(str).str.zfill(5)

print("Metadata shape :", meta.shape)
print("Columns        :", meta.columns.tolist())

# ============================================================
# 3.  Filter out SEVERE stages  (H&Y / UPDRS V  >= 4.0)
#     NaN means Healthy Control → must be preserved
# ============================================================
before = len(meta)

# Keep row if UPDRS V is NaN (HC) OR score is below 4.0
meta_filtered = meta[meta['UPDRS V'].isna() | (meta['UPDRS V'] < 4.0)].copy()

after = len(meta_filtered)

print(f"Removed {before - after} subject(s) with UPDRS V >= 4.0")
print(f"Remaining subjects: {after}")   # should now be ~72

# Quick sanity check
print("\nUPDRS V distribution after filter:")
print(meta_filtered['UPDRS V'].value_counts(dropna=False).sort_index())

# ============================================================
# 4.  Assign group labels
#       • Healthy Controls : UPDRS V is NaN  (no H&Y score)
#       • Early PD         : UPDRS V  1.0 – 2.0
#       • Moderate PD      : UPDRS V  2.5 – 3.0
# ============================================================
def assign_group(row):
    score = row['UPDRS V']

    # Healthy controls have no UPDRS V score → NaN
    if pd.isna(score):
        return 'Healthy Control'
    elif 1.0 <= score <= 2.0:
        return 'Early PD'
    elif 2.5 <= score <= 3.0:
        return 'Moderate PD'
    else:
        return 'Unknown'   # safety catch

meta_filtered['Group'] = meta_filtered.apply(assign_group, axis=1)

print("Group distribution:")
print(meta_filtered['Group'].value_counts())
print()
print(meta_filtered[['ID', 'Disease', 'UPDRS V', 'Group']].head(72).to_string(index=False))

# ============================================================
# 5.  Parse a single SVC file
# ============================================================
def parse_svc(filepath: str) -> pd.DataFrame:
    """
    SVC format
    ----------
    Line 1   : number of samples (integer)
    Lines 2+ : Y  X  timestamp  button_state  azimuth  altitude  pressure
    """
    with open(filepath, 'r') as fh:
        lines = fh.read().splitlines()

    n_samples = int(lines[0].strip())
    data_lines = lines[1: n_samples + 1]          # safety slice

    records = []
    for line in data_lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        records.append({
            'y'            : float(parts[0]),
            'x'            : float(parts[1]),
            'timestamp'    : float(parts[2]),
            'button_state' : int(parts[3]),
            'azimuth'      : float(parts[4]),
            'altitude'     : float(parts[5]),
            'pressure'     : float(parts[6]),
        })

    df = pd.DataFrame(records)
    df['n_declared'] = n_samples
    return df

# ============================================================
# 6.  Load ALL SVC files  (task number only, no session)
#     Filename format:  {subjectID}_{task}_{repetition}
#     e.g.  00098_6_1  →  subject=00098, task=6
# ============================================================
all_records = []
valid_ids   = set(meta_filtered['ID'].tolist())

for subj_folder in sorted(Path(SVC_ROOT).iterdir()):
    if not subj_folder.is_dir():
        continue

    subj_id = subj_folder.name.zfill(5)   # e.g. '98' → '00098'

    if subj_id not in valid_ids:
        continue

    meta_row = meta_filtered[meta_filtered['ID'] == subj_id].iloc[0]

    svc_files = sorted([f for f in subj_folder.iterdir() if f.is_file()])

    for svc_path in svc_files:
        try:
            svc_df = parse_svc(str(svc_path))
        except Exception as e:
            print(f"  [WARN] Could not parse {svc_path.name}: {e}")
            continue

        # e.g. '00098_6_1' → parts = ['00098', '6', '1']
        parts = svc_path.stem.split('_')
        task  = parts[1] if len(parts) > 1 else 'unknown'
        # parts[2] is just the repetition suffix ('1'), not used

        svc_df['subject_id'] = subj_id
        svc_df['task']       = task          # task number (1-8)
        svc_df['file_name']  = svc_path.name

        # Attach metadata
        svc_df['group']      = meta_row['Group']
        svc_df['updrs_v']    = meta_row['UPDRS V']
        svc_df['disease']    = meta_row['Disease']
        svc_df['age']        = meta_row['Age']
        svc_df['sex']        = meta_row['Sex']

        all_records.append(svc_df)

full_df = pd.concat(all_records, ignore_index=True)

print(f"Total stroke-point rows : {len(full_df):,}")
print(f"Unique subjects         : {full_df['subject_id'].nunique()}")
print(f"Unique SVC files        : {full_df['file_name'].nunique()}")
# print(f"\nTasks found             : {sorted(full_df['task'].unique())}")

# ============================================================
# 4.2  Patient-Level Stratified 5-Fold Cross-Validation Setup
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold

# --- Define two classification objectives ---

# Objective 1: Detection (Healthy Control vs PD)
def assign_detection_label(group):
    if group == 'Healthy Control':
        return 0
    else:
        return 1  # Early PD or Moderate PD

# Objective 2: Staging (Early PD vs Moderate PD) — only PD subjects
def assign_staging_label(group):
    if group == 'Early PD':
        return 0
    elif group == 'Moderate PD':
        return 1
    else:
        return None  # Healthy controls excluded

# --- Build a subject-level dataframe for fold assignment ---
subject_df = meta_filtered[['ID', 'Group']].copy()
subject_df['detection_label'] = subject_df['Group'].apply(assign_detection_label)
subject_df['staging_label'] = subject_df['Group'].apply(assign_staging_label)

print("=" * 60)
print("DETECTION TASK — Label Distribution (subject-level)")
print("=" * 60)
print(subject_df['detection_label'].value_counts().rename({0: 'Healthy (0)', 1: 'PD (1)'}))
print()

staging_subject_df = subject_df.dropna(subset=['staging_label']).copy()
staging_subject_df['staging_label'] = staging_subject_df['staging_label'].astype(int)
print("=" * 60)
print("STAGING TASK — Label Distribution (subject-level)")
print("=" * 60)
print(staging_subject_df['staging_label'].value_counts().rename({0: 'Early PD (0)', 1: 'Moderate PD (1)'}))
print()

# --- Create the 5-Fold splitter ---
N_SPLITS = 5
RANDOM_STATE = 42

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- Build file-level dataframes for each task ---
# Each SVC file is one sample; the group key is subject_id

file_level_df = full_df.groupby('file_name').first().reset_index()[
    ['file_name', 'subject_id', 'task', 'group', 'updrs_v', 'disease']
]
file_level_df['detection_label'] = file_level_df['group'].apply(assign_detection_label)
file_level_df['staging_label'] = file_level_df['group'].apply(assign_staging_label)

# Detection file-level
detect_file_df = file_level_df.copy()

# Staging file-level (PD only)
stage_file_df = file_level_df.dropna(subset=['staging_label']).copy()
stage_file_df['staging_label'] = stage_file_df['staging_label'].astype(int)

print(f"Detection samples (files): {len(detect_file_df)}")
print(f"Staging samples (files)  : {len(stage_file_df)}")
print()

# --- Generate and display fold splits ---
def generate_folds(df, label_col, task_name):
    """Generate fold indices and print summary."""
    X = df.index.values
    y = df[label_col].values
    groups = df['subject_id'].values

    folds = []
    print(f"{'='*60}")
    print(f"Fold Summary for: {task_name}")
    print(f"{'='*60}")
    for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
        train_subjects = set(df.iloc[train_idx]['subject_id'].unique())
        test_subjects = set(df.iloc[test_idx]['subject_id'].unique())
        overlap = train_subjects & test_subjects

        print(f"\nFold {fold_idx + 1}:")
        print(f"  Train: {len(train_idx)} files, {len(train_subjects)} subjects")
        print(f"  Test : {len(test_idx)} files, {len(test_subjects)} subjects")
        print(f"  Train label dist: {dict(pd.Series(y[train_idx]).value_counts().sort_index())}")
        print(f"  Test  label dist: {dict(pd.Series(y[test_idx]).value_counts().sort_index())}")
        print(f"  Patient leakage : {'NONE ✓' if len(overlap) == 0 else f'WARNING: {overlap}'}")

        folds.append((train_idx, test_idx))
    return folds

detection_folds = generate_folds(detect_file_df, 'detection_label', 'DETECTION')
staging_folds = generate_folds(stage_file_df, 'staging_label', 'STAGING')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Metadata shape : (75, 11)
Columns        : ['ID', 'Nationality', 'Sex', 'Disease', 'PD status', 'Age', 'Dominant hand', 'LED', 'UPDRS V', 'Length of PD', 'Unnamed: 10']


In [ ]:
# ============================================================
# 7.  FEATURE EXTRACTION
# ============================================================
from scipy import stats
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq

class FeatureExtractor:
    """
    Extracts handwriting features from a single SVC recording.
    Compatible with both PaHaW (SVC) and UCI Spiral (CSV).
    """

    def __init__(self):
        pass

    def _safe(self, arr, func, default=0.0):
        """Safely compute a stat."""
        if len(arr) == 0:
            return default
        try:
            val = func(arr)
            return default if (np.isnan(val) or np.isinf(val)) else float(val)
        except:
            return default

    def _kinematics(self, x, y, timestamps):
        """Compute velocity, acceleration, jerk from x, y, timestamps."""
        dt = np.diff(timestamps)
        dt[dt == 0] = 1e-6

        dx = np.diff(x)
        dy = np.diff(y)

        vx = dx / dt
        vy = dy / dt
        velocity = np.sqrt(vx**2 + vy**2)

        if len(velocity) > 1:
            dt2 = dt[1:]
            ax = np.diff(vx) / dt2
            ay = np.diff(vy) / dt2
            acceleration = np.sqrt(ax**2 + ay**2)
        else:
            acceleration = np.array([0.0])

        if len(acceleration) > 1:
            dt3 = dt[2:len(acceleration)]
            if len(dt3) == 0:
                dt3 = np.array([dt[0]])
            jerk = np.diff(acceleration) / dt3[:len(np.diff(acceleration))]
        else:
            jerk = np.array([0.0])

        return velocity, acceleration, jerk, vx, vy, dt

    def _freq_features(self, signal_data, sr, prefix):
        """Frequency-domain features from a signal."""
        feats = {}
        if len(signal_data) < 10:
            for key in ["tremor_power_ratio", "normal_power_ratio",
                        "tremor_to_normal", "dominant_freq",
                        "spectral_entropy", "spectral_centroid",
                        "spectral_bandwidth", "high_freq_ratio"]:
                feats[f"{prefix}_{key}"] = 0.0
            return feats

        fft_vals = np.abs(rfft(signal_data - np.mean(signal_data)))
        freqs = rfftfreq(len(signal_data), d=1.0 / sr)
        psd = fft_vals ** 2
        total = np.sum(psd) + 1e-10

        tremor = (freqs >= 3) & (freqs <= 8)
        normal = (freqs >= 0.5) & (freqs < 3)
        high   = (freqs >= 8) & (freqs <= 15)

        feats[f"{prefix}_tremor_power_ratio"] = np.sum(psd[tremor]) / total
        feats[f"{prefix}_normal_power_ratio"] = np.sum(psd[normal]) / total
        feats[f"{prefix}_high_freq_ratio"]    = np.sum(psd[high]) / total
        feats[f"{prefix}_tremor_to_normal"]   = np.sum(psd[tremor]) / (np.sum(psd[normal]) + 1e-10)

        if len(fft_vals) > 1:
            feats[f"{prefix}_dominant_freq"] = freqs[np.argmax(fft_vals[1:]) + 1]
        else:
            feats[f"{prefix}_dominant_freq"] = 0.0

        psd_n = psd / (total + 1e-10)
        psd_n = psd_n[psd_n > 0]
        feats[f"{prefix}_spectral_entropy"]   = -np.sum(psd_n * np.log2(psd_n + 1e-15))
        feats[f"{prefix}_spectral_centroid"]  = np.sum(freqs * fft_vals) / (np.sum(fft_vals) + 1e-10)
        feats[f"{prefix}_spectral_bandwidth"] = np.sqrt(
            np.sum(((freqs - feats[f"{prefix}_spectral_centroid"])**2) * fft_vals)
            / (np.sum(fft_vals) + 1e-10)
        )

        return feats

    def extract(self, df):
        """
        Extract features from a single SVC recording DataFrame.

        Expected columns: x, y, timestamp, pressure, button_state,
                         azimuth, altitude
        """
        f = {}

        x = df["x"].values.astype(float)
        y = df["y"].values.astype(float)
        pressure = df["pressure"].values.astype(float)
        ts = df["timestamp"].values.astype(float)
        button = df["button_state"].values.astype(int) if "button_state" in df.columns else None

        # Normalize pressure to 0–1 if needed
        if pressure.max() > 1.0:
            pressure = pressure / pressure.max()

        # Convert timestamps to seconds if in ms
        if ts.max() > 10000:
            ts = ts / 1000.0

        # Estimate sampling rate
        dt_median = np.median(np.diff(ts))
        sr = 1.0 / dt_median if dt_median > 0 else 100.0

        # ── Kinematics ────────────────────────────────
        velocity, acceleration, jerk, vx, vy, dt = self._kinematics(x, y, ts)

        # ══════════════════════════════════════════════
        # 1. PRESSURE FEATURES (10)
        # ══════════════════════════════════════════════
        f["pressure_mean"]     = self._safe(pressure, np.mean)
        f["pressure_std"]      = self._safe(pressure, np.std)
        f["pressure_max"]      = self._safe(pressure, np.max)
        f["pressure_min"]      = self._safe(pressure, np.min)
        f["pressure_range"]    = f["pressure_max"] - f["pressure_min"]
        f["pressure_cv"]       = f["pressure_std"] / (f["pressure_mean"] + 1e-6)
        f["pressure_skew"]     = self._safe(pressure, stats.skew)
        f["pressure_kurtosis"] = self._safe(pressure, stats.kurtosis)
        f["pressure_iqr"]      = self._safe(pressure, lambda p: np.percentile(p, 75) - np.percentile(p, 25))
        f["pressure_median"]   = self._safe(pressure, np.median)

        # ══════════════════════════════════════════════
        # 2. VELOCITY FEATURES (10)
        # ══════════════════════════════════════════════
        f["velocity_mean"]     = self._safe(velocity, np.mean)
        f["velocity_std"]      = self._safe(velocity, np.std)
        f["velocity_max"]      = self._safe(velocity, np.max)
        f["velocity_min"]      = self._safe(velocity, np.min)
        f["velocity_range"]    = f["velocity_max"] - f["velocity_min"]
        f["velocity_cv"]       = f["velocity_std"] / (f["velocity_mean"] + 1e-6)
        f["velocity_skew"]     = self._safe(velocity, stats.skew)
        f["velocity_kurtosis"] = self._safe(velocity, stats.kurtosis)
        f["velocity_iqr"]      = self._safe(velocity, lambda v: np.percentile(v, 75) - np.percentile(v, 25))
        f["velocity_median"]   = self._safe(velocity, np.median)

        # ══════════════════════════════════════════════
        # 3. ACCELERATION FEATURES (8)
        # ══════════════════════════════════════════════
        f["accel_mean"]     = self._safe(acceleration, np.mean)
        f["accel_std"]      = self._safe(acceleration, np.std)
        f["accel_max"]      = self._safe(acceleration, lambda a: np.max(np.abs(a)))
        f["accel_cv"]       = f["accel_std"] / (f["accel_mean"] + 1e-6)
        f["accel_skew"]     = self._safe(acceleration, stats.skew)
        f["accel_kurtosis"] = self._safe(acceleration, stats.kurtosis)
        f["accel_rms"]      = self._safe(acceleration, lambda a: np.sqrt(np.mean(a**2)))
        f["accel_iqr"]      = self._safe(acceleration, lambda a: np.percentile(a, 75) - np.percentile(a, 25))

        # ══════════════════════════════════════════════
        # 4. JERK FEATURES (5)
        # ══════════════════════════════════════════════
        f["jerk_mean"]     = self._safe(jerk, lambda j: np.mean(np.abs(j)))
        f["jerk_std"]      = self._safe(jerk, np.std)
        f["jerk_max"]      = self._safe(jerk, lambda j: np.max(np.abs(j)))
        f["jerk_cv"]       = f["jerk_std"] / (f["jerk_mean"] + 1e-6)
        f["jerk_skew"]     = self._safe(jerk, stats.skew)

        # ══════════════════════════════════════════════
        # 5. SPATIAL / GEOMETRY FEATURES (8)
        # ══════════════════════════════════════════════
        f["stroke_width"]    = np.ptp(x)
        f["stroke_height"]   = np.ptp(y)
        f["stroke_area"]     = f["stroke_width"] * f["stroke_height"]
        f["aspect_ratio"]    = f["stroke_width"] / (f["stroke_height"] + 1e-6)
        f["path_length"]     = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
        f["path_efficiency"] = (
            np.sqrt((x[-1] - x[0])**2 + (y[-1] - y[0])**2)
            / (f["path_length"] + 1e-6)
        )

        # Micrographia
        n = len(x)
        seg = max(n // 4, 10)
        start_spread = np.std(x[:seg]) + np.std(y[:seg])
        end_spread   = np.std(x[-seg:]) + np.std(y[-seg:])
        f["micrographia_ratio"] = end_spread / (start_spread + 1e-6)

        # Mean curvature
        if len(x) > 2:
            gx  = np.gradient(x)
            gy  = np.gradient(y)
            ggx = np.gradient(gx)
            ggy = np.gradient(gy)
            curv = np.abs(gx * ggy - gy * ggx) / ((gx**2 + gy**2)**1.5 + 1e-10)
            f["mean_curvature"] = self._safe(curv, np.mean)
        else:
            f["mean_curvature"] = 0.0

        # ══════════════════════════════════════════════
        # 6. FREQUENCY / TREMOR FEATURES (16)
        # ══════════════════════════════════════════════
        freq_x   = self._freq_features(x - np.mean(x), sr, "x")
        freq_vel = self._freq_features(velocity, sr, "vel") if len(velocity) > 10 else {
            f"vel_{k}": 0.0 for k in [
                "tremor_power_ratio", "normal_power_ratio", "high_freq_ratio",
                "tremor_to_normal", "dominant_freq", "spectral_entropy",
                "spectral_centroid", "spectral_bandwidth",
            ]
        }
        f.update(freq_x)
        f.update(freq_vel)

        # ══════════════════════════════════════════════
        # 7. TEMPORAL PATTERN FEATURES (12)
        # ══════════════════════════════════════════════

        # Zero crossing rates
        if len(acceleration) > 1:
            f["zcr_accel"] = np.sum(np.diff(np.sign(acceleration - np.mean(acceleration))) != 0) / len(acceleration)
        else:
            f["zcr_accel"] = 0.0

        if len(velocity) > 1:
            f["zcr_velocity"] = np.sum(np.diff(np.sign(velocity - np.mean(velocity))) != 0) / len(velocity)
        else:
            f["zcr_velocity"] = 0.0

        # Autocorrelations
        f["autocorr_pressure"] = np.corrcoef(pressure[:-1], pressure[1:])[0, 1] if len(pressure) > 1 else 0.0
        f["autocorr_velocity"] = np.corrcoef(velocity[:-1], velocity[1:])[0, 1] if len(velocity) > 1 else 0.0
        f["autocorr_accel"]    = np.corrcoef(acceleration[:-1], acceleration[1:])[0, 1] if len(acceleration) > 1 else 0.0

        # Fix NaN autocorrelations
        for key in ["autocorr_pressure", "autocorr_velocity", "autocorr_accel"]:
            if np.isnan(f[key]) or np.isinf(f[key]):
                f[key] = 0.0

        # Peak counts and rates
        duration = (ts[-1] - ts[0]) if len(ts) > 1 else 1.0
        duration = max(duration, 1e-6)

        vel_peaks = len(find_peaks(velocity)[0]) if len(velocity) > 2 else 0
        prs_peaks = len(find_peaks(pressure)[0]) if len(pressure) > 2 else 0

        f["velocity_peak_count"]  = vel_peaks
        f["pressure_peak_count"]  = prs_peaks
        f["velocity_peak_rate"]   = vel_peaks / duration
        f["pressure_peak_rate"]   = prs_peaks / duration

        # Signal entropy
        for name, sig in [("pressure", pressure), ("velocity", velocity)]:
            if len(sig) > 1:
                hist, _ = np.histogram(sig, bins=30, density=True)
                hist = hist[hist > 0]
                f[f"{name}_entropy"] = -np.sum(hist * np.log2(hist + 1e-15))
            else:
                f[f"{name}_entropy"] = 0.0

        # ══════════════════════════════════════════════
        # 8. PEN UP/DOWN FEATURES (4)
        # ══════════════════════════════════════════════
        if button is not None:
            on_surface = button == 1
            in_air     = button == 0
            f["on_surface_ratio"]    = np.mean(on_surface)
            f["in_air_ratio"]        = np.mean(in_air)
            f["num_pen_lifts"]       = int(np.sum(np.diff(button.astype(int)) == -1))
            f["mean_stroke_dur"]     = (np.sum(on_surface) / (f["num_pen_lifts"] + 1)) / sr
        else:
            f["on_surface_ratio"]  = 1.0
            f["in_air_ratio"]      = 0.0
            f["num_pen_lifts"]     = 0
            f["mean_stroke_dur"]   = duration

        # ══════════════════════════════════════════════
        # 9. DURATION FEATURES (3)
        # ══════════════════════════════════════════════
        f["signal_length"]    = len(df)
        f["duration_seconds"] = duration
        f["sampling_rate"]    = sr

        # ══════════════════════════════════════════════
        # 10. AZIMUTH & ALTITUDE FEATURES (6)
        # ══════════════════════════════════════════════
        if "azimuth" in df.columns:
            az = df["azimuth"].values.astype(float)
            f["azimuth_mean"]  = self._safe(az, np.mean)
            f["azimuth_std"]   = self._safe(az, np.std)
            f["azimuth_range"] = np.ptp(az)
        else:
            f["azimuth_mean"]  = 0.0
            f["azimuth_std"]   = 0.0
            f["azimuth_range"] = 0.0

        if "altitude" in df.columns:
            alt = df["altitude"].values.astype(float)
            f["altitude_mean"]  = self._safe(alt, np.mean)
            f["altitude_std"]   = self._safe(alt, np.std)
            f["altitude_range"] = np.ptp(alt)
        else:
            f["altitude_mean"]  = 0.0
            f["altitude_std"]   = 0.0
            f["altitude_range"] = 0.0

        # ══════════════════════════════════════════════
        # CLEANUP
        # ══════════════════════════════════════════════
        for key in f:
            if np.isnan(f[key]) or np.isinf(f[key]):
                f[key] = 0.0

        return f


# ============================================================
# 8.  EXTRACT FEATURES FROM ALL SVC FILES
# ============================================================

print("=" * 60)
print("EXTRACTING FEATURES FROM ALL SVC FILES")
print("=" * 60)

extractor = FeatureExtractor()

feature_records = []
skipped = 0

for file_name in detect_file_df['file_name'].values:
    # Get the raw data for this file
    file_data = full_df[full_df['file_name'] == file_name].copy()

    if len(file_data) < 50:
        skipped += 1
        continue

    try:
        features = extractor.extract(file_data)
        features['file_name'] = file_name
        feature_records.append(features)
    except Exception as e:
        print(f"  [WARN] Error extracting {file_name}: {e}")
        skipped += 1

feature_df = pd.DataFrame(feature_records)

print(f"\n✅ Extracted features from {len(feature_df)} files")
print(f"⚠️ Skipped {skipped} files")
print(f"📋 Feature count: {len(feature_df.columns) - 1}")  # -1 for file_name

# Merge with file-level metadata
feature_df = feature_df.merge(
    detect_file_df[['file_name', 'subject_id', 'task', 'group',
                     'detection_label', 'staging_label']],
    on='file_name',
    how='left',
)

print(f"\n📊 Detection label distribution:")
print(feature_df['detection_label'].value_counts().rename({0: 'Healthy (0)', 1: 'PD (1)'}))

staging_feature_df = feature_df.dropna(subset=['staging_label']).copy()
staging_feature_df['staging_label'] = staging_feature_df['staging_label'].astype(int)
print(f"\n📊 Staging label distribution:")
print(staging_feature_df['staging_label'].value_counts().rename({0: 'Early PD (0)', 1: 'Moderate PD (1)'}))


# ============================================================
# 9.  DEFINE FEATURE COLUMNS (exclude metadata)
# ============================================================
META_COLS = ['file_name', 'subject_id', 'task', 'group',
             'detection_label', 'staging_label']

FEATURE_COLS = [c for c in feature_df.columns if c not in META_COLS]

print(f"\n📋 Total features for modeling: {len(FEATURE_COLS)}")
print(f"   Features: {FEATURE_COLS}")


# ============================================================
# 10. TRAIN & VALIDATE — 5-FOLD CV (Detection + Staging)
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score,
)
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, LSTM, Conv1D, Dropout, Reshape,
    BatchNormalization, GlobalAveragePooling1D,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(RANDOM_STATE)


# ── Model Creators ────────────────────────────────────────

def create_rf():
    return RandomForestClassifier(
        n_estimators=300, max_depth=12,
        min_samples_split=5, min_samples_leaf=2,
        max_features="sqrt", class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    )

def create_xgb(n_classes):
    return xgb.XGBClassifier(
        n_estimators=300, max_depth=6,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, eval_metric="mlogloss" if n_classes > 2 else "logloss",
        use_label_encoder=False,
    )

def create_lstm(n_features, n_classes):
    model = Sequential([
        Reshape((n_features, 1), input_shape=(n_features,)),
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        BatchNormalization(),
        LSTM(64),
        Dropout(0.3),
        BatchNormalization(),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

def create_cnn(n_features, n_classes):
    model = Sequential([
        Reshape((n_features, 1), input_shape=(n_features,)),
        Conv1D(64, 3, activation='relu', padding='same'),
        BatchNormalization(),
        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        Dropout(0.3),
        Conv1D(64, 3, activation='relu', padding='same'),
        GlobalAveragePooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


# ── Training & Evaluation Function ────────────────────────

def run_cv_pipeline(data_df, folds, label_col, feature_cols, task_name, n_classes):
    """
    Run complete 5-Fold CV for all models on a given task.
    """
    print(f"\n{'═' * 70}")
    print(f"  {task_name}")
    print(f"{'═' * 70}")

    model_configs = {
        "Random Forest": {"type": "ml", "fn": create_rf},
        "XGBoost":       {"type": "ml", "fn": lambda: create_xgb(n_classes)},
        "LSTM":          {"type": "dl", "fn": lambda: create_lstm(len(feature_cols), n_classes)},
        "1D-CNN":        {"type": "dl", "fn": lambda: create_cnn(len(feature_cols), n_classes)},
    }

    all_results = {}

    for model_name, cfg in model_configs.items():
        print(f"\n  ── {model_name} {'─' * (50 - len(model_name))}")

        fold_metrics = {"accuracy": [], "f1": [], "precision": [], "recall": []}
        all_y_true, all_y_pred, all_y_prob = [], [], []

        for fold_idx, (train_idx, val_idx) in enumerate(folds):
            # Get data
            X_train = data_df.iloc[train_idx][feature_cols].values
            X_val   = data_df.iloc[val_idx][feature_cols].values
            y_train = data_df.iloc[train_idx][label_col].values
            y_val   = data_df.iloc[val_idx][label_col].values

            # Scale
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_val_s   = scaler.transform(X_val)

            # Train & Predict
            if cfg["type"] == "ml":
                model = cfg["fn"]()
                model.fit(X_train_s, y_train)
                y_pred = model.predict(X_val_s)
                y_prob = model.predict_proba(X_val_s)

            elif cfg["type"] == "dl":
                model = cfg["fn"]()
                y_train_oh = to_categorical(y_train, n_classes)
                y_val_oh   = to_categorical(y_val, n_classes)

                callbacks = [
                    EarlyStopping(patience=15, restore_best_weights=True, verbose=0),
                    ReduceLROnPlateau(factor=0.5, patience=7, verbose=0),
                ]
                model.fit(
                    X_train_s, y_train_oh,
                    validation_data=(X_val_s, y_val_oh),
                    epochs=150, batch_size=8,
                    callbacks=callbacks, verbose=0,
                )
                y_prob = model.predict(X_val_s, verbose=0)
                y_pred = np.argmax(y_prob, axis=1)

            # Metrics
            acc  = accuracy_score(y_val, y_pred)
            f1   = f1_score(y_val, y_pred, average='weighted', zero_division=0)
            prec = precision_score(y_val, y_pred, average='weighted', zero_division=0)
            rec  = recall_score(y_val, y_pred, average='weighted', zero_division=0)

            fold_metrics["accuracy"].append(acc)
            fold_metrics["f1"].append(f1)
            fold_metrics["precision"].append(prec)
            fold_metrics["recall"].append(rec)

            all_y_true.extend(y_val)
            all_y_pred.extend(y_pred)
            all_y_prob.extend(y_prob)

            print(f"     Fold {fold_idx+1}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f}")

        # Summary
        print(f"\n     📊 MEAN ± STD:")
        for m, vals in fold_metrics.items():
            print(f"        {m:<12}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

        # Confusion matrix
        cm = confusion_matrix(all_y_true, all_y_pred)
        print(f"\n     Confusion Matrix:\n{cm}")
        print(f"\n     Classification Report:")
        if task_name == "DETECTION":
            target = ["Healthy", "PD"]
        else:
            target = ["Early PD", "Moderate PD"]
        print(classification_report(all_y_true, all_y_pred,
                                    target_names=target, zero_division=0))

        all_results[model_name] = {
            "fold_metrics": fold_metrics,
            "mean": {k: np.mean(v) for k, v in fold_metrics.items()},
            "std":  {k: np.std(v) for k, v in fold_metrics.items()},
            "y_true": all_y_true,
            "y_pred": all_y_pred,
            "y_prob": all_y_prob,
        }

    return all_results


# ── Run Detection CV ──────────────────────────────────────
detection_results = run_cv_pipeline(
    data_df=detect_file_df.merge(feature_df[['file_name'] + FEATURE_COLS], on='file_name'),
    folds=detection_folds,
    label_col='detection_label',
    feature_cols=FEATURE_COLS,
    task_name="DETECTION (Healthy vs PD)",
    n_classes=2,
)

# ── Run Staging CV ────────────────────────────────────────
staging_merged = stage_file_df.merge(feature_df[['file_name'] + FEATURE_COLS], on='file_name')

staging_results = run_cv_pipeline(
    data_df=staging_merged,
    folds=staging_folds,
    label_col='staging_label',
    feature_cols=FEATURE_COLS,
    task_name="STAGING (Early PD vs Moderate PD)",
    n_classes=2,
)


# ============================================================
# 11. SELECT BEST MODELS & RETRAIN ON ALL DATA
# ============================================================
import joblib, json

MODELS_DIR = '/content/drive/MyDrive/teasis/models'
os.makedirs(MODELS_DIR, exist_ok=True)

def retrain_and_save(data_df, label_col, feature_cols, results, task_tag, n_classes):
    """Retrain best ML and DL models on ALL data and save."""

    # Find best ML and DL
    ml_models = {k: v for k, v in results.items() if k in ["Random Forest", "XGBoost"]}
    dl_models = {k: v for k, v in results.items() if k in ["LSTM", "1D-CNN"]}

    best_ml = max(ml_models, key=lambda k: ml_models[k]["mean"]["f1"])
    best_dl = max(dl_models, key=lambda k: dl_models[k]["mean"]["f1"])

    print(f"\n🏆 Best ML ({task_tag}): {best_ml} (F1: {ml_models[best_ml]['mean']['f1']:.4f})")
    print(f"🏆 Best DL ({task_tag}): {best_dl} (F1: {dl_models[best_dl]['mean']['f1']:.4f})")

    X_all = data_df[feature_cols].values
    y_all = data_df[label_col].values

    # Final scaler
    final_scaler = StandardScaler()
    X_scaled = final_scaler.fit_transform(X_all)

    # ── Retrain best ML ───────────────────────────────
    if best_ml == "Random Forest":
        final_ml = create_rf()
    else:
        final_ml = create_xgb(n_classes)
    final_ml.fit(X_scaled, y_all)

    # ── Retrain best DL ───────────────────────────────
    if best_dl == "LSTM":
        final_dl = create_lstm(len(feature_cols), n_classes)
    else:
        final_dl = create_cnn(len(feature_cols), n_classes)

    y_oh = to_categorical(y_all, n_classes)
    callbacks = [
        EarlyStopping(patience=20, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(factor=0.5, patience=10, verbose=0),
    ]
    final_dl.fit(X_scaled, y_oh, epochs=200, batch_size=8,
                 callbacks=callbacks, validation_split=0.1, verbose=1)

    # ── Save ──────────────────────────────────────────
    tag = task_tag.lower()

    joblib.dump(final_ml, os.path.join(MODELS_DIR, f"{tag}_best_ml.pkl"))
    final_dl.save(os.path.join(MODELS_DIR, f"{tag}_best_dl.h5"))
    joblib.dump(final_scaler, os.path.join(MODELS_DIR, f"{tag}_scaler.pkl"))

    # Save metadata
    meta_save = {
        "task": task_tag,
        "best_ml": best_ml,
        "best_dl": best_dl,
        "n_features": len(feature_cols),
        "n_classes": n_classes,
        "feature_names": feature_cols,
        "cv_results": {
            name: {"mean": res["mean"], "std": res["std"]}
            for name, res in results.items()
        },
    }
    with open(os.path.join(MODELS_DIR, f"{tag}_metadata.json"), "w") as f:
        json.dump(meta_save, f, indent=2)

    print(f"\n✅ Saved {task_tag} models to {MODELS_DIR}/")
    return best_ml, best_dl


# ── Detection ─────────────────────────────────────────────
detect_merged = detect_file_df.merge(feature_df[['file_name'] + FEATURE_COLS], on='file_name')

best_detect_ml, best_detect_dl = retrain_and_save(
    detect_merged, 'detection_label', FEATURE_COLS,
    detection_results, "DETECTION", n_classes=2,
)

# ── Staging ───────────────────────────────────────────────
staging_merged = stage_file_df.merge(feature_df[['file_name'] + FEATURE_COLS], on='file_name')

best_stage_ml, best_stage_dl = retrain_and_save(
    staging_merged, 'staging_label', FEATURE_COLS,
    staging_results, "STAGING", n_classes=2,
)

# ── Save label encoder ────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

detect_le = LabelEncoder()
detect_le.fit(['Healthy', 'PD'])
joblib.dump(detect_le, os.path.join(MODELS_DIR, "detection_label_encoder.pkl"))

stage_le = LabelEncoder()
stage_le.fit(['Early PD', 'Moderate PD'])
joblib.dump(stage_le, os.path.join(MODELS_DIR, "staging_label_encoder.pkl"))

# ── Save feature names ────────────────────────────────────
with open(os.path.join(MODELS_DIR, "feature_names.json"), "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f"\n{'═' * 60}")
print("ALL MODELS SAVED SUCCESSFULLY!")
print(f"{'═' * 60}")
print(f"\nSaved to: {MODELS_DIR}/")
print(f"  detection_best_ml.pkl")
print(f"  detection_best_dl.h5")
print(f"  detection_scaler.pkl")
print(f"  detection_metadata.json")
print(f"  staging_best_ml.pkl")
print(f"  staging_best_dl.h5")
print(f"  staging_scaler.pkl")
print(f"  staging_metadata.json")
print(f"  detection_label_encoder.pkl")
print(f"  staging_label_encoder.pkl")
print(f"  feature_names.json")


# ============================================================
# 12. UCI SPIRAL LOADER (for the deployed system)
# ============================================================

def load_uci_spiral_file(filepath_or_buffer, separator=";"):
    """
    Load a single UCI Spiral CSV file and convert to
    PaHaW-compatible format.

    UCI columns: x, y, z, pressure(0-1023), grip_angle, time(ms), test_id
    Output:      x, y, pressure(0-1), azimuth, altitude, timestamp(s), button_state
    """
    UCI_COLS = ["x", "y", "z", "pressure", "grip_angle", "time", "test_id"]

    # Read file
    if isinstance(filepath_or_buffer, str):
        df = pd.read_csv(filepath_or_buffer, sep=separator, header=None, names=UCI_COLS)
    else:
        df = pd.read_csv(filepath_or_buffer, sep=separator, header=None, names=UCI_COLS)

    # Try comma if semicolon didn't work
    if len(df.columns) == 1:
        if isinstance(filepath_or_buffer, str):
            df = pd.read_csv(filepath_or_buffer, sep=",", header=None, names=UCI_COLS)
        else:
            filepath_or_buffer.seek(0)
            df = pd.read_csv(filepath_or_buffer, sep=",", header=None, names=UCI_COLS)

    # ── Convert to PaHaW-compatible format ────────────
    result = pd.DataFrame()
    result["x"]            = df["x"]
    result["y"]            = df["y"]
    result["pressure"]     = df["pressure"] / 1023.0              # Normalize to 0–1
    result["azimuth"]      = df["grip_angle"]                     # Similar to PaHaW azimuth
    result["altitude"]     = df["z"]                              # Similar to PaHaW altitude
    result["timestamp"]    = df["time"] / 1000.0                  # ms → seconds
    result["button_state"] = (df["z"] == 0).astype(int)           # On surface = 1
    result["test_id"]      = df["test_id"]

    return result


def predict_uci_file(filepath_or_buffer, task="detection", model_type="ml"):
    """
    Complete prediction pipeline for a UCI Spiral file.

    Args:
        filepath_or_buffer: Path to CSV or uploaded file buffer
        task: "detection" or "staging"
        model_type: "ml" or "dl"

    Returns:
        dict with prediction, confidence, probabilities, features
    """
    # ── Load & convert ────────────────────────────────
    uci_df = load_uci_spiral_file(filepath_or_buffer)

    # ── Split by test_id if multiple tests ────────────
    # Predict on each test separately, return all results
    results = []

    for test_id in uci_df["test_id"].unique():
        test_df = uci_df[uci_df["test_id"] == test_id].copy()
        test_df = test_df.drop(columns=["test_id"])

        if len(test_df) < 50:
            continue

        # ── Extract features (SAME extractor as PaHaW) ─
        ext = FeatureExtractor()
        features = ext.extract(test_df)

        # ── Load model artifacts ──────────────────────
        tag = task.lower()
        model_path  = os.path.join(MODELS_DIR, f"{tag}_best_{model_type}.pkl"
                      if model_type == "ml" else f"{tag}_best_{model_type}.h5")
        scaler      = joblib.load(os.path.join(MODELS_DIR, f"{tag}_scaler.pkl"))
        feat_names  = json.load(open(os.path.join(MODELS_DIR, "feature_names.json")))

        # ── Arrange features in correct order ─────────
        X = pd.DataFrame([features])[feat_names]
        X = X.fillna(0).replace([np.inf, -np.inf], 0)

        # ── Scale ─────────────────────────────────────
        X_scaled = scaler.transform(X.values)

        # ── Predict ───────────────────────────────────
        if model_type == "ml":
            model = joblib.load(model_path)
            y_pred = model.predict(X_scaled)[0]
            y_prob = model.predict_proba(X_scaled)[0]
        else:
            model = tf.keras.models.load_model(model_path)
            y_prob = model.predict(X_scaled, verbose=0)[0]
            y_pred = np.argmax(y_prob)

        # ── Map back to label ─────────────────────────
        if task == "detection":
            class_names = ["Healthy", "PD"]
        else:
            class_names = ["Early PD", "Moderate PD"]

        test_name = {0: "Static Spiral", 1: "Dynamic Spiral", 2: "Stability Test"}

        results.append({
            "test_type":     test_name.get(test_id, f"Test {test_id}"),
            "prediction":    class_names[y_pred],
            "confidence":    float(y_prob[y_pred]),
            "probabilities": {class_names[i]: float(y_prob[i]) for i in range(len(y_prob))},
            "features":      features,
        })

    return results


# ============================================================
# 13. QUICK VERIFICATION — Test the UCI pipeline
# ============================================================

print(f"\n{'═' * 60}")
print("UCI SPIRAL INTEGRATION — READY")
print(f"{'═' * 60}")
print(f"""
Your system is now ready!

WHAT'S BEEN DONE:
  ✅ PaHaW loaded and preprocessed (SVC files)
  ✅ Features extracted ({len(FEATURE_COLS)} features)
  ✅ 5-Fold Stratified Group CV (no patient leakage)
  ✅ Detection task: Healthy vs PD
  ✅ Staging task: Early PD vs Moderate PD
  ✅ 4 models trained: RF, XGBoost, LSTM, 1D-CNN
  ✅ Best models retrained on all data & saved
  ✅ UCI Spiral loader ready

WHAT THE USER DOES:
  1. Upload a UCI Spiral CSV file to the system
  2. System auto-converts to PaHaW format
  3. System extracts SAME features
  4. System predicts using trained models
  5. User sees: "Early PD — 87% confidence"

TO TEST ON UCI:
  results = predict_uci_file("path/to/uci_patient01.csv",
                              task="detection", model_type="ml")
  for r in results:
      print(f"  {{r['test_type']}}: {{r['prediction']}} ({{r['confidence']:.1%}})")
""")